# KOSIS 조기 BGE 검색 + HCX 구조화 (5,000건 통합본)

`is_claim_news_5000_true.csv`를 업로드한 뒤 위에서부터 순서대로 실행합니다.

- BGE-M3와 reranker: GPU 사용
- KOSIS 메타 조회와 HCX API: GPU를 거의 사용하지 않음
- 모든 결과는 Google Drive에 저장되며 중단 후 다시 실행하면 이어받음

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=True)

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/repro-baseline-20260727'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INPUT_DIR = DRIVE_ROOT / 'inputs'
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
RUN_DIR = DRIVE_ROOT / 'runs' / 'early_bge_rag_5000'

INPUT_CSV = INPUT_DIR / 'is_claim_news_5000_true.csv'
CANDIDATES_CSV = RUN_DIR / 'early_bge_candidates_top20.csv'
CONTEXT_CSV = RUN_DIR / 'early_bge_context_top5.csv'
UNIQUE_TABLES_CSV = RUN_DIR / 'early_bge_unique_top5_tables.csv'
META_CSV = RUN_DIR / 'early_bge_meta_index.csv'
HCX_OUTPUT_CSV = RUN_DIR / 'hcx_early_bge_extracted.csv'
READY_CSV = RUN_DIR / 'hcx_early_bge_kosis_ready.csv'
REJECTED_CSV = RUN_DIR / 'hcx_early_bge_kosis_rejected.csv'

# 0이면 업로드한 is_claim=True 전체를 처리합니다.
HCX_LIMIT = 0

INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('입력:', INPUT_CSV)
print('결과:', RUN_DIR)

## 1. 코드 준비

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH,
        REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<3',
    'sentence-transformers>=3.4,<6',
    'transformers>=4.45,<6',
    'requests>=2.31,<3',
    'python-dotenv>=1.0,<2',
], check=True)
print('코드 준비 완료:', REPO_DIR)

## 2. `is_claim_news_5000_true.csv` 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()
expected_name = 'is_claim_news_5000_true.csv'
if expected_name not in uploaded:
    raise RuntimeError(f'{expected_name} 파일을 선택하세요.')

INPUT_CSV.write_bytes(uploaded[expected_name])
print('업로드 완료:', INPUT_CSV)

In [ ]:
import pandas as pd

required_index_files = [
    INDEX_DIR / 'manifest.json',
    INDEX_DIR / 'tables.csv',
    INDEX_DIR / 'embeddings.npy',
]
missing = [str(path) for path in required_index_files if not path.exists()]
if missing:
    raise FileNotFoundError('BGE 인덱스 파일이 없습니다: ' + ', '.join(missing))

input_df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
if 'claim_id' not in input_df.columns or 'is_claim' not in input_df.columns:
    raise RuntimeError('입력 CSV에 claim_id 또는 is_claim 컬럼이 없습니다.')

print('입력 행:', len(input_df))
print('고유 claim:', input_df['claim_id'].nunique())
print('BGE 인덱스: 준비 완료')

## 3. BGE-M3 Top-20 검색 + reranker

In [ ]:
search_command = [
    sys.executable, '-u', str(REPO_DIR / 'kosis_early_retrieve.py'),
    '--input', str(INPUT_CSV),
    '--output-candidates', str(CANDIDATES_CSV),
    '--output-context', str(CONTEXT_CSV),
    '--semantic-index', str(INDEX_DIR),
    '--semantic-top-k', '20',
    '--rerank-top-k', '20',
    '--context-top-k', '5',
    '--checkpoint-every', '10',
    '--device', 'cuda',
]
subprocess.run(search_command, check=True)

In [ ]:
candidates = pd.read_csv(CANDIDATES_CSV, encoding='utf-8-sig')
contexts = pd.read_csv(CONTEXT_CSV, encoding='utf-8-sig')

top5 = candidates[candidates['candidate_rank'] <= 5].copy()
unique_tables = top5.drop_duplicates(['org_id', 'tbl_id'])[
    ['org_id', 'tbl_id', 'tbl_name', 'category_path']
]
unique_tables.to_csv(UNIQUE_TABLES_CSV, index=False, encoding='utf-8-sig')

print('검색 claim:', contexts['claim_id'].nunique())
print('Top-20 후보 행:', len(candidates))
print('고유 Top-5 통계표:', len(unique_tables))

## 4. KOSIS 공식 메타 보강

Colab 보안 비밀에 `KOSIS_API_KEY`를 등록하세요. 이 단계는 GPU를 사용하지 않습니다.

In [ ]:
from google.colab import userdata

if not os.environ.get('KOSIS_API_KEY'):
    os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY') or ''
if not os.environ.get('KOSIS_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')

# 이전 실행의 메타가 있으면 복사해 기존 조회 결과를 재사용합니다.
old_meta = DRIVE_ROOT / 'runs' / 'early_bge_rag' / 'early_bge_meta_index.csv'
if not META_CSV.exists() and old_meta.exists():
    shutil.copy2(old_meta, META_CSV)
    print('기존 KOSIS 메타 캐시 재사용:', old_meta)

meta_command = [
    sys.executable, '-u', str(REPO_DIR / 'kosis_build_meta_index.py'),
    '--table-index', str(UNIQUE_TABLES_CSV),
    '--out', str(META_CSV),
    '--delay', '0.15',
    '--resume',
]
subprocess.run(meta_command, check=True)

In [ ]:
enrich_command = [
    sys.executable, '-u', str(REPO_DIR / 'kosis_early_retrieve.py'),
    '--reuse-candidates', str(CANDIDATES_CSV),
    '--output-candidates', str(CANDIDATES_CSV),
    '--output-context', str(CONTEXT_CSV),
    '--meta-index', str(META_CSV),
    '--context-top-k', '5',
]
subprocess.run(enrich_command, check=True)

contexts = pd.read_csv(CONTEXT_CSV, encoding='utf-8-sig')
print('메타 보강 context:', len(contexts))

## 5. HCX 측정값 구조화

업로드한 `is_claim=True` 전체를 처리합니다. 중단 후 다시 실행하면 완료된 claim을 건너뛰고 이어받습니다.

In [ ]:
if not os.environ.get('CLOVA_API_KEY'):
    os.environ['CLOVA_API_KEY'] = userdata.get('CLOVA_API_KEY') or ''
if not os.environ.get('CLOVA_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 CLOVA_API_KEY를 등록하세요.')

hcx_command = [
    sys.executable, '-u', str(REPO_DIR / 'extract_hcx.py'),
    '--input', str(INPUT_CSV),
    '--retrieval-context', str(CONTEXT_CSV),
    '--output', str(HCX_OUTPUT_CSV),
    '--model', 'HCX-007',
    '--limit', str(HCX_LIMIT),
    '--sleep', '0.5',
]
subprocess.run(hcx_command, check=True)

## 6. READY 게이트와 결과 요약

In [ ]:
gate_command = [
    sys.executable, '-u', str(REPO_DIR / 'prepare_kosis_mapping_input.py'),
    '--input', str(HCX_OUTPUT_CSV),
    '--output', str(READY_CSV),
    '--rejected-output', str(REJECTED_CSV),
]
subprocess.run(gate_command, check=True)

result_files = {
    'BGE candidates': CANDIDATES_CSV,
    'retrieval context': CONTEXT_CSV,
    'unique tables': UNIQUE_TABLES_CSV,
    'KOSIS meta': META_CSV,
    'HCX extracted': HCX_OUTPUT_CSV,
    'READY': READY_CSV,
    'rejected': REJECTED_CSV,
}

for name, file_path in result_files.items():
    if file_path.exists():
        frame = pd.read_csv(file_path, encoding='utf-8-sig')
        claim_count = frame['claim_id'].nunique() if 'claim_id' in frame.columns else '-'
        print(f'{name:20s}: rows={len(frame):,}, claims={claim_count}')
    else:
        print(f'{name:20s}: 없음')

rejected = pd.read_csv(REJECTED_CSV, encoding='utf-8-sig')
if 'mapping_exclusion_code' in rejected.columns:
    display(rejected['mapping_exclusion_code'].value_counts(dropna=False))

print('완료. 결과 폴더:', RUN_DIR)